# ElectroBench Vita VPK on Vita3K (Kaggle, free GPU)

Runs `ElectroBench-vita.vpk` (the instrumented build, commit `d15844d`) inside Vita3K
on a headless GPU notebook, then harvests the port's persistent spinlog
(`ux0:/data/ELEB00002/spinlog.txt`, including the `[VGL-EB]` init-stage traces) and
screenshots of whatever the emulator displayed.

**Setup (once):**
1. **Runtime → Change runtime type → GPU** (T4). Every cell below is timeout-guarded.
2. Optional: attach a Kaggle dataset containing your own `PSVUPDAT.PUP` (Add data →
   Create/upload dataset). If none is found, the firmware is downloaded from Sony's
   public update CDN automatically (verified working, `SCEUF` header).
3. **Run → Run all cells.** Order matters; each cell builds on the previous one.

Artifacts land in `/kaggle/working/output/` (logs) and `/kaggle/working/shots/`
(screenshots) — download them from the file panel or via Save Version.

**What this does and does not prove:** Vita3K is HLE — `sceAppMgrGetBudgetInfo`
and friends are emulated, so the free-memory numbers will differ from your retail
Vita. A green run here means the GL stack works and gives a *baseline* spinlog to
diff against your device log; the device remains authoritative for the memory-drain
bug.

In [ ]:
# --- system deps (headless X display, Vulkan probe, screenshots) ---
import os, subprocess
r = subprocess.run(
    "apt-get update -qq >/tmp/apt.log 2>&1 && "
    "apt-get install -y -qq xauth xvfb vulkan-tools imagemagick libvulkan1 unzip curl >>/tmp/apt.log 2>&1; "
    "rc=$?; tail -5 /tmp/apt.log; exit $rc",
    shell=True, executable="/bin/bash")
print("apt rc:", r.returncode)
assert r.returncode == 0, "dependency install failed — see /tmp/apt.log above"

# SDL has no audio device in a notebook; give it a dummy driver up front so
# Vita3K's SDL init cannot fail on audio.
os.environ["SDL_AUDIODRIVER"] = "dummy"
print("deps ok, SDL_AUDIODRIVER=dummy")

In [ ]:
# --- obtain Vita3K (official continuous build) ---
import glob, shutil, urllib.request, zipfile

WORK = "/kaggle/working"
VITADIR = os.path.join(WORK, "vita3k")
os.makedirs(VITADIR, exist_ok=True)

def sh(cmd, timeout=300):
    return subprocess.run(cmd, shell=True, executable="/bin/bash",
                          capture_output=True, text=True, timeout=timeout)

def find_binary(root):
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f in ("Vita3K", "AppRun"):
                return os.path.join(dirpath, f)
    return None

def try_zip():
    zp = os.path.join(WORK, "ubuntu-latest.zip")
    if not os.path.exists(zp):
        print("downloading ubuntu-latest.zip ...")
        urllib.request.urlretrieve(
            "https://github.com/Vita3K/Vita3K/releases/download/continuous/ubuntu-latest.zip", zp)
    zipfile.ZipFile(zp).extractall(VITADIR)
    return find_binary(VITADIR)

def try_appimage():
    ai = os.path.join(WORK, "Vita3K-x86_64.AppImage")
    if not os.path.exists(ai):
        print("downloading Vita3K-x86_64.AppImage ...")
        urllib.request.urlretrieve(
            "https://github.com/Vita3K/Vita3K/releases/download/continuous/Vita3K-x86_64.AppImage", ai)
    os.chmod(ai, 0o755)
    sh(f'cd "{VITADIR}" && "{ai}" --appimage-extract >/dev/null 2>&1')
    return find_binary(VITADIR)

BIN = find_binary(VITADIR) or try_zip()

def probe_works(binary):
    p = sh(f'timeout 60 xvfb-run -a "{binary}" --version 2>&1 || true')
    out = p.stdout + p.stderr
    print("probe:", out.strip()[:300] or "(no output)")
    return "error while loading shared libraries" not in out

if BIN is None or not probe_works(BIN):
    print("zip build unusable, trying AppImage fallback ...")
    BIN = try_appimage()

assert BIN and os.path.exists(BIN), "could not obtain a Vita3K binary"
os.chmod(BIN, 0o755)
probe_works(BIN)
print("Vita3K at:", BIN)

In [ ]:
# --- obtain the VPK (public GitHub raw first, attached dataset as fallback) ---
VPK = None
VPK_URL = "https://raw.githubusercontent.com/deadbeef7/ElectroBench/vitaport/ElectroBench-vita.vpk"
cand = os.path.join(WORK, "ElectroBench-vita.vpk")
try:
    print("trying GitHub raw ...")
    urllib.request.urlretrieve(VPK_URL, cand)
    names = zipfile.ZipFile(cand).namelist()   # raises on an HTML error page
    assert "eboot.bin" in names, "no eboot.bin in downloaded zip"
    VPK = cand
except Exception as e:
    print("GitHub raw failed:", e)

if VPK is None:
    hits = [p for p in glob.glob("/kaggle/input/**/*.vpk", recursive=True)]
    if hits:
        VPK = hits[0]
    else:
        raise FileNotFoundError(
            "ElectroBench-vita.vpk not found. Attach it as a Kaggle dataset "
            "(Add data -> Create dataset -> upload the .vpk) and re-run this cell.")

print("VPK:", VPK, os.path.getsize(VPK), "bytes")

In [ ]:
# --- GPU / Vulkan sanity (fail fast with a clear message if no GPU runtime) ---
r = sh("nvidia-smi -L 2>&1 || true")
print(r.stdout.strip() or r.stderr.strip())
r = sh("timeout 120 vulkaninfo --summary 2>&1 | head -50 || true", timeout=150)
print(r.stdout)
if "deviceName" not in r.stdout and "GPU" not in r.stdout:
    print("\n*** No Vulkan device visible. Use Runtime -> Change runtime type -> GPU, then restart the session. ***")
else:
    print("Vulkan device present — Vita3K will use it via its Xlib surface (works under Xvfb).")

In [ ]:
# --- firmware: Vita3K cannot boot any title without it ---
import urllib.parse

def collect_pups():
    out = []
    for base in ("/kaggle/input", WORK):
        for p in glob.glob(os.path.join(base, "**", "*"), recursive=True):
            if os.path.isfile(p) and p.lower().endswith(".pup"):
                out.append(p)
    return out

pups = collect_pups()
if pups:
    print("using PUP from dataset:", *pups, sep="\n  ")
else:
    print("no dataset PUP — downloading from Sony's public update CDN ...")
    urls = [
        "http://dus01.psp2.update.playstation.net/update/psp2/image/2019_0924/"
        "sd_8b5f60b56c3da8365b973dba570c53a5/PSP2UPDAT.PUP?dest=us",
        "http://dus01.psv.update.playstation.net/update/psv/image/2022_0209/"
        "rel_f2c7b12fe85496ec88a0391b514d6e3b/PSVUPDAT.PUP",
    ]
    fwdir = os.path.join(WORK, "firmware")
    os.makedirs(fwdir, exist_ok=True)
    for u in urls:
        name = os.path.basename(urllib.parse.urlparse(u).path)
        dest = os.path.join(fwdir, name)
        try:
            print("  downloading", name, "...")
            urllib.request.urlretrieve(u, dest)
            with open(dest, "rb") as f:
                magic = f.read(4)
            if magic != b"SCEUF":
                print("    bad magic", magic, "-> discarding")
                os.remove(dest)
                continue
            print("    ok:", os.path.getsize(dest), "bytes")
            pups.append(dest)
        except Exception as e:
            print("    failed:", e)

if not pups:
    raise RuntimeError(
        "No firmware available. Attach your PSVUPDAT.PUP as a dataset "
        "(Vita3K cannot boot titles without firmware).")

for p in pups:
    print("=== installing firmware:", p)
    r = sh(f'timeout 900 xvfb-run -a -s "-screen 0 1280x720x24" '
           f'"{BIN}" --firmware "{p}" 2>&1 | tail -25 || true', timeout=960)
    print(r.stdout[-2000:])
print("firmware step done (timeouts during GUI phase are harmless)")

In [ ]:
# --- renderer: force Vulkan (NVIDIA ICD exposes Xlib surface under Xvfb) ---
sh(f'timeout 20 xvfb-run -a "{BIN}" >/dev/null 2>&1 || true', timeout=40)  # generate default config

cfg = os.path.expanduser("~/.config/Vita3K/config.yml")
EXTRA = ""
if os.path.exists(cfg):
    lines = open(cfg).read().splitlines()
    changed = False
    for i, ln in enumerate(lines):
        if ln.strip().startswith("backend-renderer"):
            lines[i] = "backend-renderer: Vulkan"
            changed = True
    if changed:
        open(cfg, "w").write("\n".join(lines) + "\n")
        print("config: backend-renderer forced to Vulkan")
    else:
        print("config: no backend-renderer key (Vita3K will auto-pick)")
else:
    print("config not created yet — relying on auto-detect")

help_out = sh(f'timeout 60 xvfb-run -a "{BIN}" --help 2>&1 || true', timeout=90)
if "backend-renderer" in help_out.stdout:
    EXTRA = "--backend-renderer Vulkan"
print("CLI renderer arg:", EXTRA or "(none — config/default only)")
print(sh(f'grep -n "backend-renderer" "{cfg}" 2>/dev/null || echo "key not present"').stdout)

In [ ]:
# --- install the VPK ---
r = sh(f'timeout 240 xvfb-run -a -s "-screen 0 960x544x24" '
       f'"{BIN}" {EXTRA} "{VPK}" 2>&1 | tail -40 || true', timeout=300)
print(r.stdout)

app_dirs = glob.glob(os.path.expanduser("~/.local/share/**/ELEB00002"), recursive=True)
if not app_dirs:
    print("positional install not confirmed, retrying with the sdl frontend ...")
    r = sh(f'timeout 240 xvfb-run -a -s "-screen 0 960x544x24" '
           f'"{BIN}" --frontend sdl {EXTRA} "{VPK}" 2>&1 | tail -20 || true', timeout=300)
    print(r.stdout)
    app_dirs = glob.glob(os.path.expanduser("~/.local/share/**/ELEB00002"), recursive=True)

installed = bool(app_dirs)
print("installed app dir:", app_dirs or "NOT FOUND (run cell 9 anyway — it falls back to install-and-run)")

In [ ]:
# --- run the homebrew and grab screenshots while it runs ---
shots = os.path.join(WORK, "shots")
os.makedirs(shots, exist_ok=True)

if installed:
    run_args = f'--frontend sdl -r ELEB00002 {EXTRA}'
else:
    run_args = f'{EXTRA} "{VPK}"'
    print("WARNING: app dir not found; launching positional install-and-run")
print("running:", run_args)

script = f'''timeout 170 xvfb-run -a -s "-screen 0 960x544x24" bash -c '
  ( for i in $(seq 1 16); do sleep 8; import -window root "{shots}/shot_$(printf %02d $i).png" 2>/dev/null; done ) &
  SC=$!
  timeout 165 "{BIN}" {run_args} 2>&1 | tail -30
  kill $SC 2>/dev/null
'
''' 
r = sh(script, timeout=240)
print(r.stdout[-3000:])
if r.stderr.strip():
    print("stderr:", r.stderr[-800:])
print("screenshots:", sorted(os.path.basename(p) for p in glob.glob(os.path.join(shots, "*.png"))))

In [ ]:
# --- harvest logs: the port's spinlog (THE artifact) + Vita3K's own log ---
OUT = os.path.join(WORK, "output")
os.makedirs(OUT, exist_ok=True)

roots = [os.path.expanduser("~/.local/share/Vita3K"),
         os.path.expanduser("~/.cache/Vita3K"),
         os.path.expanduser("~/.config/Vita3K")]

collected = []
for root in roots:
    if not os.path.isdir(root):
        continue
    for p in glob.glob(os.path.join(root, "**", "*"), recursive=True):
        if not os.path.isfile(p):
            continue
        base = os.path.basename(p).lower()
        if "spinlog" in base or "log" in base or "ELEB00002" in p:
            dst = os.path.join(OUT, os.path.basename(p))
            n = 1
            while os.path.exists(dst):
                dst = os.path.join(OUT, f"{n}_{os.path.basename(p)}")
                n += 1
            shutil.copy2(p, dst)
            collected.append(dst)

print("collected files:")
for c in collected:
    print(" ", c, os.path.getsize(c), "bytes")

spins = glob.glob(os.path.join(roots[0], "**", "spinlog.txt"), recursive=True)
if spins:
    print("\n================ SPINLOG", spins[0], "================")
    print(open(spins[0], errors="replace").read())
else:
    print("\n*** no spinlog found — the port's code never ran (or never got as far as VitaLog). ***")

logs = []
if os.path.isdir(roots[1]):
    logs = [p for p in glob.glob(os.path.join(roots[1], "**", "*"), recursive=True)
            if os.path.isfile(p) and "log" in os.path.basename(p).lower()]
if logs:
    print("\n================ VITA3K LOG (tail):", logs[0], "================")
    lines = open(logs[0], errors="replace").read().splitlines()
    print("\n".join(lines[-80:]))
else:
    print("\n(no Vita3K log found under", roots[1], ")")

In [ ]:
# --- show what the emulator displayed ---
from IPython.display import Image, display
shots = sorted(glob.glob(os.path.join(WORK, "shots", "*.png")))
if not shots:
    print("no screenshots captured")
for s in shots:
    display(Image(filename=s))

## Troubleshooting

- **`vulkaninfo` shows no device / black screenshots** → the session has no GPU:
  Runtime → Change runtime type → GPU, then restart and re-run from the top.
- **Vita3K pops an error and hangs** → every launch is wrapped in `timeout`, so the
  notebook always proceeds to log collection. The Vita3K log tail shows what it said.
- **Renderer problems** → cell 7 forces `Vulkan`. If Vulkan itself fails on this
  runtime, change that line to `backend-renderer: OpenGL` (llvmpipe under Xvfb —
  slow, but it can boot the app).
- **Firmware download fails** (Sony CDN unreachable from the region) → download
  `PSVUPDAT.PUP` yourself (Vita3K's own *Help → Download Firmware* points at the same
  public file) and attach it as a dataset; the notebook picks it up automatically.
- **Fonts** → the quickstart also mentions the firmware font package. ElectroBench's
  scenes bake their own glyph atlas, so it should render without it.
- **Kaggle quota** → free tier ≈ 30 GPU-hours/week; sessions also disconnect when
  idle, so keep the tab alive while it runs.
- **Reading the result** → compare the harvested spinlog's `[VGL-EB]` stage lines
  against the device log: `vgl init ok via …` under Vita3K + a device run still
  failing pins the bug to retail-firmware specifics (memory budgets, sysapp mode).